# Polymer Property Prediction — Tg / Egc — Colab Pipeline

RDKit descriptors + fingerprints + **polyBERT embeddings** (Tier 1: extra features,
Tier 2: independent regression head) blended with LightGBM + XGBoost via an
NNLS stacking meta-model.

**How to run:** Runtime → Change runtime type → CPU is fine (no GPU required).
Run cells top to bottom. Upload `train.csv` / `test.csv` when prompted in the
data-loading cell, or point `TRAIN_PATH`/`TEST_PATH` at Drive paths.


In [ ]:
# Colab ships with most of the scientific stack; we only need to add these.
!pip install -q rdkit lightgbm xgboost optuna sentence-transformers scipy


In [ ]:
import io, os, sys, pickle
from datetime import datetime

import numpy as np
import optuna
import pandas as pd
from joblib import Parallel, delayed
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, rdMolDescriptors
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from scipy.optimize import nnls
from xgboost import XGBRegressor

RDLogger.DisableLog("rdApp.*")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Paths (Colab-friendly: everything under /content) ──────────────────────
BASE_DIR    = "/content"
DATA_DIR    = f"{BASE_DIR}/data"
OUTPUT_DIR  = f"{BASE_DIR}/outputs"
TRAIN_PATH  = f"{DATA_DIR}/train.csv"
TEST_PATH   = f"{DATA_DIR}/test.csv"
OUTPUT_PATH = f"{OUTPUT_DIR}/submission.csv"
PBERT_TRAIN_CACHE = f"{OUTPUT_DIR}/polybert_train.pkl"
PBERT_TEST_CACHE  = f"{OUTPUT_DIR}/polybert_test.pkl"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Run-size knobs (reduce further if a Colab session is time-limited) ─────
N_FOLDS         = 5
SEEDS           = [42, 0, 123]     # multi-seed averaging for the FINAL fit only
TUNE_SEED       = SEEDS[0]         # hyperparameters are tuned ONCE, then reused
                                    # across all seeds -- retuning per seed was
                                    # the single biggest source of wasted runtime
                                    # in the original pipeline; the data doesn't
                                    # change between seeds, only the CV shuffle does.
N_TUNE_FOLDS    = 3                # fewer folds during tuning; full N_FOLDS for final scoring
EARLY_STOPPING_ROUNDS = 50
N_TRIALS        = 20               # LGB Optuna trials per target
N_XGB_TRIALS    = 8                # XGB Optuna trials per target
TUNE_N_ESTIMATORS  = 1000
FINAL_N_ESTIMATORS = 3000
PROBE_N_ESTIMATORS = 300
N_CHAIN_UNITS   = 3

POLYBERT_MODEL   = "kuelumbus/polyBERT"
POLYBERT_HEAD    = "ridge"         # "ridge" or "mlp"

print(f"Config loaded. Working dir: {BASE_DIR}")


In [ ]:
# Upload train.csv / test.csv if they aren't already at TRAIN_PATH / TEST_PATH.
# (If you mounted Google Drive instead, just point TRAIN_PATH/TEST_PATH at
# your Drive paths in the config cell above and skip this upload step.)
if not (os.path.exists(TRAIN_PATH) and os.path.exists(TEST_PATH)):
    from google.colab import files
    print("Please upload train.csv and test.csv")
    uploaded = files.upload()
    for fname in uploaded:
        dest = f"{DATA_DIR}/{fname}"
        os.rename(fname, dest)
        print(f"  saved -> {dest}")
else:
    print("train.csv / test.csv already present, skipping upload.")


In [ ]:
# ── Feature computation (RDKit descriptors, fingerprints, physics-motivated features) ──

def _desc_one(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return {name: np.nan for name, _ in Descriptors._descList}
    return Descriptors.CalcMolDescriptors(mol)

def _ecfp4_one(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return [np.nan] * 2048
    return list(AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048))

def _ecfp6_one(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return [np.nan] * 2048
    return list(AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=2048))

def _maccs_one(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return [np.nan] * 167
    return list(MACCSkeys.GenMACCSKeys(mol))

def _topo_one(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return {"star_distance": np.nan, "star_distance_frac": np.nan}
    star_idx = [a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == "*"]
    if len(star_idx) != 2:
        return {"star_distance": np.nan, "star_distance_frac": np.nan}
    dmat = Chem.GetDistanceMatrix(mol)
    star_dist = dmat[star_idx[0], star_idx[1]]
    diameter  = dmat.max()
    return {
        "star_distance"     : star_dist,
        "star_distance_frac": star_dist / diameter if diameter > 0 else 0.0,
    }

def _electronic_one(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return {
            "num_aromatic_rings": np.nan, "aromatic_atom_fraction": np.nan,
            "num_rotatable_bonds": np.nan, "sp2_atom_fraction": np.nan,
            "num_nonarom_double_bonds": np.nan,
        }
    n_atoms = mol.GetNumAtoms()
    n_arom  = sum(1 for a in mol.GetAtoms() if a.GetIsAromatic())
    n_sp2   = sum(1 for a in mol.GetAtoms()
                  if a.GetHybridization() == Chem.rdchem.HybridizationType.SP2)
    n_dbl   = sum(1 for b in mol.GetBonds()
                  if b.GetBondTypeAsDouble() == 2.0 and not b.GetIsAromatic())
    return {
        "num_aromatic_rings"      : rdMolDescriptors.CalcNumAromaticRings(mol),
        "aromatic_atom_fraction"  : n_arom / n_atoms if n_atoms > 0 else 0.0,
        "num_rotatable_bonds"     : rdMolDescriptors.CalcNumRotatableBonds(mol),
        "sp2_atom_fraction"       : n_sp2 / n_atoms if n_atoms > 0 else 0.0,
        "num_nonarom_double_bonds": n_dbl,
    }

def _tg_specific_one(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return {"backbone_rotatable_bonds": np.nan}
    star_idx = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() == 0]
    if len(star_idx) != 2:
        return {"backbone_rotatable_bonds": np.nan}
    path = Chem.GetShortestPath(mol, star_idx[0], star_idx[1])
    rot = 0
    for i in range(len(path) - 1):
        bond = mol.GetBondBetweenAtoms(path[i], path[i + 1])
        if bond.GetBondTypeAsDouble() == 1.0 and not bond.IsInRing():
            rot += 1
    return {"backbone_rotatable_bonds": rot}

def _conjugation_one(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return {"max_conjugation_path": np.nan}
    sp2 = {a.GetIdx() for a in mol.GetAtoms()
           if a.GetHybridization() == Chem.rdchem.HybridizationType.SP2}
    if not sp2:
        return {"max_conjugation_path": 0}
    visited, max_comp = set(), 0
    for start in sp2:
        if start in visited:
            continue
        comp, stack = set(), [start]
        while stack:
            node = stack.pop()
            if node in comp:
                continue
            comp.add(node)
            for bond in mol.GetAtomWithIdx(node).GetBonds():
                nbr = bond.GetOtherAtomIdx(node)
                if nbr in sp2 and nbr not in comp:
                    stack.append(nbr)
        visited |= comp
        max_comp = max(max_comp, len(comp))
    return {"max_conjugation_path": max_comp}

def _build_chain(smi, n_units=N_CHAIN_UNITS):
    base = Chem.MolFromSmiles(smi)
    if base is None:
        return None
    base_stars = sorted(a.GetIdx() for a in base.GetAtoms() if a.GetAtomicNum() == 0)
    if len(base_stars) != 2:
        return None
    left_star_base, right_star_base = base_stars
    if (base.GetAtomWithIdx(left_star_base).GetDegree() != 1 or
            base.GetAtomWithIdx(right_star_base).GetDegree() != 1):
        return None

    chain = Chem.RWMol(base)
    chain_right_star = right_star_base

    for _ in range(n_units - 1):
        offset       = chain.GetNumAtoms()
        right_nbr    = chain.GetAtomWithIdx(chain_right_star).GetNeighbors()[0].GetIdx()
        bond_type    = chain.GetBondBetweenAtoms(chain_right_star, right_nbr).GetBondType()

        new_left_star  = left_star_base  + offset
        new_right_star = right_star_base + offset
        new_left_nbr   = base.GetAtomWithIdx(left_star_base).GetNeighbors()[0].GetIdx() + offset

        chain = Chem.RWMol(Chem.CombineMols(chain.GetMol(), base))
        chain.AddBond(right_nbr, new_left_nbr, bond_type)

        for idx in sorted([chain_right_star, new_left_star], reverse=True):
            chain.RemoveAtom(idx)

        n_below = sum(1 for idx in [chain_right_star, new_left_star] if idx < new_right_star)
        chain_right_star = new_right_star - n_below

    for idx in sorted((a.GetIdx() for a in chain.GetAtoms() if a.GetAtomicNum() == 0), reverse=True):
        chain.RemoveAtom(idx)

    try:
        Chem.SanitizeMol(chain)
        return Chem.MolToSmiles(chain.GetMol())
    except Exception:
        return None

def compute_features(df):
    smiles = df["smiles"].tolist()
    chain_raw = [_build_chain(s) for s in smiles]
    n_fail    = sum(1 for c in chain_raw if c is None)
    chain_smi = [c if c is not None else s for c, s in zip(chain_raw, smiles)]
    if n_fail:
        print(f"    Chain build failures: {n_fail}/{len(smiles)} (fell back to monomer)")

    descs  = Parallel(n_jobs=-1, prefer="threads")(delayed(_desc_one)(s)        for s in smiles)
    ecfp4  = Parallel(n_jobs=-1, prefer="threads")(delayed(_ecfp4_one)(s)       for s in smiles)
    ecfp6  = Parallel(n_jobs=-1, prefer="threads")(delayed(_ecfp6_one)(s)       for s in smiles)
    maccs  = Parallel(n_jobs=-1, prefer="threads")(delayed(_maccs_one)(s)       for s in smiles)
    topo   = Parallel(n_jobs=-1, prefer="threads")(delayed(_topo_one)(s)        for s in smiles)
    elec   = Parallel(n_jobs=-1, prefer="threads")(delayed(_electronic_one)(s)  for s in smiles)
    tgfeat = Parallel(n_jobs=-1, prefer="threads")(delayed(_tg_specific_one)(s) for s in smiles)
    conj   = Parallel(n_jobs=-1, prefer="threads")(delayed(_conjugation_one)(s) for s in smiles)

    ch_descs = Parallel(n_jobs=-1, prefer="threads")(delayed(_desc_one)(s)        for s in chain_smi)
    ch_ecfp4 = Parallel(n_jobs=-1, prefer="threads")(delayed(_ecfp4_one)(s)       for s in chain_smi)
    ch_ecfp6 = Parallel(n_jobs=-1, prefer="threads")(delayed(_ecfp6_one)(s)       for s in chain_smi)
    ch_maccs = Parallel(n_jobs=-1, prefer="threads")(delayed(_maccs_one)(s)       for s in chain_smi)
    ch_elec  = Parallel(n_jobs=-1, prefer="threads")(delayed(_electronic_one)(s)  for s in chain_smi)
    ch_conj  = Parallel(n_jobs=-1, prefer="threads")(delayed(_conjugation_one)(s) for s in chain_smi)

    p = f"ch{N_CHAIN_UNITS}_"

    desc_df   = pd.DataFrame(descs,   index=df.index)
    ecfp4_df  = pd.DataFrame(ecfp4,   index=df.index, columns=[f"ecfp4_{i}" for i in range(2048)])
    ecfp6_df  = pd.DataFrame(ecfp6,   index=df.index, columns=[f"ecfp6_{i}" for i in range(2048)])
    maccs_df  = pd.DataFrame(maccs,   index=df.index, columns=[f"maccs_{i}" for i in range(167)])
    topo_df   = pd.DataFrame(topo,    index=df.index)
    elec_df   = pd.DataFrame(elec,    index=df.index)
    tgfeat_df = pd.DataFrame(tgfeat,  index=df.index)
    conj_df   = pd.DataFrame(conj,    index=df.index)

    ch_desc_df  = pd.DataFrame(ch_descs, index=df.index).add_prefix(p)
    ch_ecfp4_df = pd.DataFrame(ch_ecfp4, index=df.index, columns=[f"{p}ecfp4_{i}" for i in range(2048)])
    ch_ecfp6_df = pd.DataFrame(ch_ecfp6, index=df.index, columns=[f"{p}ecfp6_{i}" for i in range(2048)])
    ch_maccs_df = pd.DataFrame(ch_maccs, index=df.index, columns=[f"{p}maccs_{i}" for i in range(167)])
    ch_elec_df  = pd.DataFrame(ch_elec,  index=df.index).add_prefix(p)
    ch_conj_df  = pd.DataFrame(ch_conj,  index=df.index).add_prefix(p)

    combined = pd.concat([
        desc_df, ecfp4_df, ecfp6_df, maccs_df, topo_df, elec_df, tgfeat_df, conj_df,
        ch_desc_df, ch_ecfp4_df, ch_ecfp6_df, ch_maccs_df, ch_elec_df, ch_conj_df,
    ], axis=1)
    combined = combined.astype(np.float32).replace([np.inf, -np.inf], np.nan)
    return combined

print("RDKit feature functions defined.")


In [ ]:
# ── polyBERT Tier 1 (frozen embeddings) + Tier 2 (embedding-only heads) ──
# polyBERT (kuelumbus/polyBERT) is a DeBERTa-v2 model pretrained on 100M
# hypothetical PSMILES strings specifically for polymer structure-property
# work. It expects PSMILES with "[*]" attachment points, which is exactly
# what canon_smiles already contains for these monomers -- no adapter needed.
# Chain (trimer) SMILES are NOT valid PSMILES here (terminal "*" atoms were
# stripped when the chain was stitched), so embeddings are computed on the
# monomer canon_smiles only.

EMBED_DIM = 600

def _load_polybert(device="cpu"):
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(POLYBERT_MODEL, device=device)

def compute_polybert_embeddings(df, smiles_col="canon_smiles", cache_path=None,
                                 batch_size=64, device="cpu"):
    smiles = df[smiles_col].tolist() if smiles_col in df.columns else df["smiles"].tolist()

    cache = {}
    if cache_path and os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)

    to_encode = sorted({s for s in smiles if s not in cache})
    if to_encode:
        print(f"  [polyBERT] encoding {len(to_encode):,} new / uncached SMILES "
              f"(of {len(set(smiles)):,} unique, {len(smiles):,} rows)...")
        model = _load_polybert(device=device)
        vecs = model.encode(to_encode, batch_size=batch_size,
                             show_progress_bar=True, convert_to_numpy=True)
        for s, v in zip(to_encode, vecs):
            cache[s] = v.astype(np.float32)
        if cache_path:
            os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
            with open(cache_path, "wb") as f:
                pickle.dump(cache, f)
    else:
        print(f"  [polyBERT] all {len(set(smiles)):,} unique SMILES already cached.")

    zero_vec = np.zeros(EMBED_DIM, dtype=np.float32)
    matrix = np.stack([cache.get(s, zero_vec) for s in smiles])
    cols = [f"pbert_{i}" for i in range(matrix.shape[1])]
    return pd.DataFrame(matrix, index=df.index, columns=cols)


def _make_head(head_type, seed, alpha=1.0):
    if head_type == "ridge":
        return Ridge(alpha=alpha, random_state=seed)
    if head_type == "mlp":
        return MLPRegressor(hidden_layer_sizes=(256, 64), alpha=alpha,
                             early_stopping=True, n_iter_no_change=15,
                             max_iter=500, random_state=seed)
    raise ValueError(f"Unknown head_type: {head_type!r}")


def _tune_alpha(X, y, cv_splits, mask_fn, seed, head_type,
                alphas=(0.1, 1.0, 3.0, 10.0, 30.0, 100.0)):
    best_alpha, best_r2 = alphas[0], -np.inf
    for alpha in alphas:
        scores = []
        for tr_idx, val_idx in cv_splits:
            m_tr, m_val = mask_fn(tr_idx), mask_fn(val_idx)
            X_tr, X_val = X.iloc[tr_idx][m_tr], X.iloc[val_idx][m_val]
            y_tr, y_val = y[tr_idx][m_tr], y[val_idx][m_val]
            scaler = StandardScaler().fit(X_tr)
            model = _make_head(head_type, seed, alpha=alpha)
            model.fit(scaler.transform(X_tr), y_tr)
            scores.append(r2_score(y_val, model.predict(scaler.transform(X_val))))
        mean_r2 = float(np.mean(scores))
        if mean_r2 > best_r2:
            best_r2, best_alpha = mean_r2, alpha
    return best_alpha


def train_polybert_heads(emb_train, y_train, strat_label, cv_splits, emb_test,
                          test_subsets, target_types=("tg", "egc"),
                          head_type="ridge", seed=42):
    n = len(y_train)
    oof = {t: np.full(n, np.nan) for t in target_types}
    test_pred = {t: np.zeros(len(test_subsets[t])) for t in target_types}
    fold_r2 = {t: [] for t in target_types}

    for ttype in target_types:
        def mask_fn(idx, _t=ttype):
            return strat_label[idx] == _t

        alpha = _tune_alpha(emb_train, y_train, cv_splits, mask_fn, seed, head_type)
        X_test_sub = emb_test.loc[test_subsets[ttype].index]

        for tr_idx, val_idx in cv_splits:
            m_tr, m_val = mask_fn(tr_idx), mask_fn(val_idx)
            X_tr, X_val = emb_train.iloc[tr_idx][m_tr], emb_train.iloc[val_idx][m_val]
            y_tr, y_val = y_train[tr_idx][m_tr], y_train[val_idx][m_val]

            scaler = StandardScaler().fit(X_tr)
            model = _make_head(head_type, seed, alpha=alpha)
            model.fit(scaler.transform(X_tr), y_tr)

            val_idx_masked = val_idx[m_val]
            pred_val = model.predict(scaler.transform(X_val))
            oof[ttype][val_idx_masked] = pred_val
            fold_r2[ttype].append(r2_score(y_val, pred_val))
            test_pred[ttype] += model.predict(scaler.transform(X_test_sub)) / len(cv_splits)

        print(f"  [polyBERT head] {ttype.upper()}  alpha={alpha}  "
              f"mean R2={np.mean(fold_r2[ttype]):+.4f}  (std {np.std(fold_r2[ttype]):.4f})")

    return {"oof": oof, "test_pred": test_pred, "fold_r2": fold_r2}


def fit_stack_meta(oof_preds, y_val):
    """NNLS meta-model over {model_name: oof_predictions} for one target type.
    Replaces a hand-tuned w/(1-w) grid with a proper, non-negative-weighted
    combination that generalizes to any number of base models."""
    names = list(oof_preds.keys())
    X = np.column_stack([oof_preds[n] for n in names])
    coef, _ = nnls(X, y_val)
    weights = dict(zip(names, coef))
    return weights

print("polyBERT Tier 1 + Tier 2 functions defined.")


In [ ]:
start = datetime.now()
print(f"Started at {start.strftime('%H:%M:%S')}\n")

print("Loading data...")
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print(f"  train: {len(train):,} rows  |  test: {len(test):,} rows")

print("\nValidating SMILES...")
train_bad = train["smiles"].apply(lambda s: Chem.MolFromSmiles(s) is None).sum()
test_bad  = test["smiles"].apply(lambda s: Chem.MolFromSmiles(s) is None).sum()
print(f"  train: {train_bad} unparseable  |  test: {test_bad} unparseable")

def _to_canon(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol) if mol is not None else smi

train["canon_smiles"] = train["smiles"].apply(_to_canon)
test["canon_smiles"]  = test["smiles"].apply(_to_canon)

dup_groups = (
    train.groupby(["canon_smiles", "target_type"])["target"]
    .agg(list).reset_index()
)
dup_groups = dup_groups[dup_groups["target"].apply(len) > 1]
print(f"\nDuplicate (canon SMILES, target_type) groups: {len(dup_groups)}")
if len(dup_groups) > 0:
    print("  [spread = max - min; large spread = labeling conflict that caps model R2]")
    for _, row in dup_groups.iterrows():
        vals   = row["target"]
        spread = max(vals) - min(vals)
        print(f"  {row['target_type'].upper()}  spread={spread:.2f}  "
              f"values={[round(v, 2) for v in vals]}  smiles={row['canon_smiles'][:50]}")

n_before = len(train)
train = (
    train.groupby(["canon_smiles", "target_type"], as_index=False)
    .agg(smiles=("smiles", "first"), target=("target", "mean"))
)
n_after = len(train)
print(f"\nDuplicate merge: {n_before:,} -> {n_after:,} rows  ({n_before - n_after} groups collapsed to mean)")


In [ ]:
print("\nComputing RDKit features...")
X_train_full = compute_features(train)
X_test_full  = compute_features(test)
print(f"  Done.  {X_train_full.shape[1]} RDKit features per molecule")

y_train     = train["target"].values
strat_label = train["target_type"].values
groups      = train["canon_smiles"].values

test_tg  = test[test["target_type"] == "tg"].copy()
test_egc = test[test["target_type"] == "egc"].copy()
test_subsets = {"tg": test_tg, "egc": test_egc}


In [ ]:
print("\nComputing polyBERT embeddings (Tier 1)...")
emb_train = compute_polybert_embeddings(train, cache_path=PBERT_TRAIN_CACHE)
emb_test  = compute_polybert_embeddings(test,  cache_path=PBERT_TEST_CACHE)
print(f"  polyBERT embeddings: {emb_train.shape[1]} dims")

# Tier 1: append embeddings as extra tabular features for the GBM blend
X_train_full = pd.concat([X_train_full, emb_train.add_prefix("t1_")], axis=1)
X_test_full  = pd.concat([X_test_full,  emb_test.add_prefix("t1_")],  axis=1)
print(f"  Combined feature matrix: {X_train_full.shape[1]} columns")


In [ ]:
print(f"\n{'='*60}")
print("  HYPERPARAMETER TUNING (once, seed={})".format(TUNE_SEED))
print(f"{'='*60}")

_tune_sgkf   = StratifiedGroupKFold(n_splits=N_TUNE_FOLDS, shuffle=True, random_state=TUNE_SEED)
_tune_splits = list(_tune_sgkf.split(X_train_full, strat_label, groups))

# Feature pruning: single probe fold, permutation importance, kept for both targets
print(f"\n  Feature pruning ({PROBE_N_ESTIMATORS}-tree probe)...")
n_orig    = X_train_full.shape[1]
keep_cols = set()
probe_tr_idx, probe_val_idx = _tune_splits[0]
for ttype in ["tg", "egc"]:
    mask_tr  = strat_label[probe_tr_idx]  == ttype
    mask_val = strat_label[probe_val_idx] == ttype
    X_probe_tr  = X_train_full.iloc[probe_tr_idx][mask_tr]
    X_probe_val = X_train_full.iloc[probe_val_idx][mask_val]
    y_probe_tr  = y_train[probe_tr_idx][mask_tr]
    y_probe_val = y_train[probe_val_idx][mask_val]

    probe = LGBMRegressor(n_estimators=PROBE_N_ESTIMATORS, num_leaves=63,
                           random_state=TUNE_SEED, n_jobs=-1, verbose=-1)
    probe.fit(X_probe_tr, y_probe_tr)
    result = permutation_importance(probe, X_probe_val, y_probe_val,
                                     n_repeats=3, random_state=TUNE_SEED, n_jobs=-1)
    imp = pd.Series(result.importances_mean, index=X_train_full.columns)
    positive = imp[imp > 0].index
    keep_cols.update(positive)
    print(f"    {ttype.upper()}: {len(positive):,} / {n_orig:,} features kept")

keep_cols = sorted(keep_cols)
X_train = X_train_full[keep_cols]
X_test  = X_test_full[keep_cols]
print(f"  Kept {len(keep_cols):,} total, dropped {n_orig - len(keep_cols):,} zero-importance")

# LGB tuning
def make_lgb_objective(ttype):
    def objective(trial):
        params = {
            "n_estimators": TUNE_N_ESTIMATORS, "random_state": TUNE_SEED,
            "n_jobs": -1, "verbose": -1,
            "num_leaves"       : trial.suggest_int("num_leaves", 15, 255),
            "learning_rate"    : trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            "min_split_gain"   : trial.suggest_float("min_split_gain", 0.0, 1.0),
            "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        }
        scores = []
        for tr_idx, val_idx in _tune_splits:
            mask_tr, mask_val = strat_label[tr_idx] == ttype, strat_label[val_idx] == ttype
            X_tr, X_val = X_train.iloc[tr_idx][mask_tr], X_train.iloc[val_idx][mask_val]
            y_tr, y_val = y_train[tr_idx][mask_tr], y_train[val_idx][mask_val]
            model = LGBMRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                      callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)])
            scores.append(r2_score(y_val, model.predict(X_val)))
        return np.mean(scores)
    return objective

best_lgb_params = {}
for ttype in ["tg", "egc"]:
    print(f"\n  Tuning LGB {ttype.upper()}...")
    study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=TUNE_SEED),
                                 pruner=optuna.pruners.MedianPruner())
    study.optimize(make_lgb_objective(ttype), n_trials=N_TRIALS, show_progress_bar=False)
    best_lgb_params[ttype] = {"random_state": TUNE_SEED, "n_jobs": -1, "verbose": -1, **study.best_params}
    print(f"    Best R2({ttype.upper()}) = {study.best_value:+.4f}")

# XGB tuning
def make_xgb_objective(ttype):
    def objective(trial):
        params = {
            "n_estimators": TUNE_N_ESTIMATORS, "random_state": TUNE_SEED,
            "n_jobs": -1, "tree_method": "hist", "device": "cpu",
            "max_depth"       : trial.suggest_int("max_depth", 3, 8),
            "learning_rate"   : trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
            "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "reg_alpha"       : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda"      : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "min_child_weight": trial.suggest_float("min_child_weight", 1, 20, log=True),
        }
        scores = []
        for tr_idx, val_idx in _tune_splits:
            mask_tr, mask_val = strat_label[tr_idx] == ttype, strat_label[val_idx] == ttype
            X_tr, X_val = X_train.iloc[tr_idx][mask_tr], X_train.iloc[val_idx][mask_val]
            y_tr, y_val = y_train[tr_idx][mask_tr], y_train[val_idx][mask_val]
            model = XGBRegressor(**params, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            scores.append(r2_score(y_val, model.predict(X_val)))
        return np.mean(scores)
    return objective

best_xgb_params = {}
for ttype in ["tg", "egc"]:
    print(f"\n  Tuning XGB {ttype.upper()}...")
    study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=TUNE_SEED),
                                 pruner=optuna.pruners.MedianPruner())
    study.optimize(make_xgb_objective(ttype), n_trials=N_XGB_TRIALS, show_progress_bar=False)
    best_xgb_params[ttype] = {"random_state": TUNE_SEED, "n_jobs": -1, "tree_method": "hist",
                               "device": "cpu", **study.best_params}
    print(f"    Best R2({ttype.upper()}) = {study.best_value:+.4f}")


In [ ]:
print(f"\n{'='*60}")
print("  PER-SEED FINAL FIT  (LGB + XGB + polyBERT head, NNLS-stacked)")
print(f"{'='*60}")

all_seed_test_tg, all_seed_test_egc, seed_cv_scores = [], [], []

for run_idx, run_seed in enumerate(SEEDS):
    print(f"\n{'#'*60}")
    print(f"  RUN {run_idx + 1}/{len(SEEDS)}  (seed={run_seed})")
    print(f"{'#'*60}")

    sgkf      = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=run_seed)
    cv_splits = list(sgkf.split(X_train, strat_label, groups))

    fold_lgb, fold_xgb, fold_y = {"tg": [], "egc": []}, {"tg": [], "egc": []}, {"tg": [], "egc": []}
    test_pred_lgb = {t: np.zeros(len(test_subsets[t])) for t in ["tg", "egc"]}
    test_pred_xgb = {t: np.zeros(len(test_subsets[t])) for t in ["tg", "egc"]}
    oof_idx = {"tg": [], "egc": []}

    for fold, (tr_idx, val_idx) in enumerate(cv_splits, 1):
        scores = {}
        for ttype in ["tg", "egc"]:
            mask_tr, mask_val = strat_label[tr_idx] == ttype, strat_label[val_idx] == ttype
            X_tr, X_val = X_train.iloc[tr_idx][mask_tr], X_train.iloc[val_idx][mask_val]
            y_tr, y_val = y_train[tr_idx][mask_tr], y_train[val_idx][mask_val]
            X_test_sub  = X_test.loc[test_subsets[ttype].index]

            lgb_params = {**best_lgb_params[ttype], "n_estimators": FINAL_N_ESTIMATORS, "random_state": run_seed}
            xgb_params = {**best_xgb_params[ttype], "n_estimators": FINAL_N_ESTIMATORS, "random_state": run_seed}

            lgb_model = LGBMRegressor(**lgb_params)
            lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                          callbacks=[early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), log_evaluation(0)])
            xgb_model = XGBRegressor(**xgb_params, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
            xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

            lgb_val, xgb_val = lgb_model.predict(X_val), xgb_model.predict(X_val)
            scores[ttype] = r2_score(y_val, (lgb_val + xgb_val) / 2)

            fold_lgb[ttype].append(lgb_val)
            fold_xgb[ttype].append(xgb_val)
            fold_y[ttype].append(y_val)
            oof_idx[ttype].append(val_idx[mask_val])
            test_pred_lgb[ttype] += lgb_model.predict(X_test_sub)
            test_pred_xgb[ttype] += xgb_model.predict(X_test_sub)

        print(f"  Fold {fold}  R2(Tg)={scores['tg']:+.4f}  R2(Egc)={scores['egc']:+.4f}  (naive 50/50)")

    # Tier 2: polyBERT-only head, same cv_splits
    print("\n  Training polyBERT Tier-2 heads...")
    pbert_result = train_polybert_heads(
        emb_train=emb_train, y_train=y_train, strat_label=strat_label,
        cv_splits=cv_splits, emb_test=emb_test, test_subsets=test_subsets,
        head_type=POLYBERT_HEAD, seed=run_seed,
    )

    # NNLS stack per target: LGB, XGB, polyBERT-head -> combined OOF + test
    print("\n  NNLS stacking (LGB + XGB + polyBERT head)...")
    seed_tg_pred, seed_egc_pred = None, None
    fold_r2_this_seed = {"tg": None, "egc": None}
    for ttype in ["tg", "egc"]:
        idx_order = np.concatenate(oof_idx[ttype])
        lgb_oof   = np.concatenate(fold_lgb[ttype])
        xgb_oof   = np.concatenate(fold_xgb[ttype])
        y_oof     = np.concatenate(fold_y[ttype])
        pbert_oof = pbert_result["oof"][ttype][idx_order]

        weights = fit_stack_meta(
            {"lgb": lgb_oof, "xgb": xgb_oof, "pbert": pbert_oof}, y_oof
        )
        stacked_oof = (weights["lgb"] * lgb_oof + weights["xgb"] * xgb_oof
                       + weights["pbert"] * pbert_oof)
        stack_r2 = r2_score(y_oof, stacked_oof)
        naive_r2 = r2_score(y_oof, 0.5 * lgb_oof + 0.5 * xgb_oof)
        print(f"    {ttype.upper()}: weights={ {k: round(v,3) for k,v in weights.items()} }  "
              f"stacked R2={stack_r2:+.4f}  (naive 50/50 LGB/XGB: {naive_r2:+.4f})")
        fold_r2_this_seed[ttype] = stack_r2

        test_lgb_avg  = test_pred_lgb[ttype]  / N_FOLDS
        test_xgb_avg  = test_pred_xgb[ttype]  / N_FOLDS
        test_pbert    = pbert_result["test_pred"][ttype]
        stacked_test  = (weights["lgb"] * test_lgb_avg + weights["xgb"] * test_xgb_avg
                         + weights["pbert"] * test_pbert)
        if ttype == "tg":
            seed_tg_pred = stacked_test
        else:
            seed_egc_pred = stacked_test

    cv_r2 = (fold_r2_this_seed["tg"] + fold_r2_this_seed["egc"]) / 2
    print(f"\n  >>> CV score (seed={run_seed}) = {cv_r2:+.4f} <<<")

    all_seed_test_tg.append(seed_tg_pred)
    all_seed_test_egc.append(seed_egc_pred)
    seed_cv_scores.append(cv_r2)


In [ ]:
print(f"\n{'='*60}")
print("  MULTI-SEED SUMMARY")
print(f"{'='*60}")
for seed, score in zip(SEEDS, seed_cv_scores):
    print(f"  Seed {seed:3d}: CV = {score:+.4f}")
print(f"  Mean CV  : {np.mean(seed_cv_scores):+.4f}  (std {np.std(seed_cv_scores):.4f})")

test_tg["target"]  = np.mean(all_seed_test_tg,  axis=0)
test_egc["target"] = np.mean(all_seed_test_egc, axis=0)

submission = pd.concat([test_tg, test_egc])[["id", "target"]].sort_values("id")
submission.to_csv(OUTPUT_PATH, index=False)

print(f"\nSubmission saved -> {OUTPUT_PATH}")
print(f"  {len(submission):,} rows  |  id range: {submission['id'].min()}-{submission['id'].max()}")
print(submission.head(5).to_string(index=False))

elapsed = datetime.now() - start
print(f"\nDone. Total time: {elapsed}")

# Download the submission file
from google.colab import files
files.download(OUTPUT_PATH)
